In [ ]:
# Save the fit result for replotting later
jwspecfit.save_result(result, "g395m_fit_result.npz")

# Export line measurements
jwspecfit.export_lines_txt(result, "g395m_lines.txt")

with open("g395m_lines.txt") as f:
    print(f.read())

## Save results and export line table

In [ ]:
fig_interactive = jwspecfit.plot_fit_interactive(result)
fig_interactive.show()

# 02 — Grating Spectrum with Broad-Line Detection

This notebook demonstrates:
1. Fitting a G395M medium-resolution grating spectrum
2. Using `fit_with_broad()` for BIC-based broad Balmer component detection
3. Comparing narrow-only vs broad models — broad components are shown
   with hatched fill in the plot

The G395M grating has R ≈ 1000 (constant), resolving [NII]–Hα–[NII] and
[SII] doublets that are blended in the prism.

In [ ]:
import jwspecfit
import matplotlib.pyplot as plt
import numpy as np

## Load the G395M spectrum

In [ ]:
spec = jwspecfit.read_fits(
    "../../data/excels-uds04-v4_g395m-f290lp_3543_63107.spec.fits",
    z=8.271,
)
print(f"Grating: {spec.grating}, {spec.n_pix} pixels")
print(f"Wave: {spec.wave_um.min():.2f} – {spec.wave_um.max():.2f} µm")

## Narrow-only fit

Bootstrap uncertainties are computed by default (200 iterations).

In [ ]:
result = jwspecfit.fit_lines(spec, z=8.271)

print(f"χ²/dof = {result.chi2:.2f}")
for name, lr in result.lines.items():
    if lr.snr > 1:
        print(f"  {name:<18s} flux={lr.flux:.2e} ± {lr.flux_err:.2e}  SNR={lr.snr:.1f}")

In [ ]:
fig = jwspecfit.plot_fit(result)
plt.show()

## Broad component detection

`fit_with_broad()` compares four models:
- **narrow**: narrow lines only
- **broad1**: narrow + intermediate broad Balmer (σ_v ≈ 3× narrow)
- **broad2**: narrow + very broad Balmer (σ_v ≈ 7× narrow)
- **both**: narrow + both broad components

Selection is by BIC with ΔBIC ≥ 6 threshold.

In [ ]:
broad_result = jwspecfit.fit_with_broad(spec, z=8.271, mode="auto")

print(f"Selected model: {broad_result.selected_model}")
print(f"BIC narrow:     {broad_result.bic_narrow:.1f}")
print(f"BIC broad1:     {broad_result.bic_broad1:.1f}")
print(f"BIC broad2:     {broad_result.bic_broad2:.1f}")
print(f"BIC both:       {broad_result.bic_both:.1f}")

In [ ]:
fig = jwspecfit.plot_fit(broad_result.best_fit)
plt.show()

## Force a broad component

Override the BIC selection with `mode="broad1"`.  The broad Gaussian
components are shown with hatched fill in the plot, so you can
distinguish them from the narrow components (solid fill).

In [ ]:
forced = jwspecfit.fit_with_broad(spec, z=8.271, mode="broad1")

# Show broad line parameters
for name, lr in forced.best_fit.lines.items():
    if "BROAD" in name:
        print(f"{name}: flux={lr.flux:.2e} ± {lr.flux_err:.2e}, σ={lr.sigma_A:.1f} Å")

fig = jwspecfit.plot_fit(forced.best_fit)
plt.show()

## Second G395M spectrum

In [ ]:
spec2 = jwspecfit.read_fits(
    "../../data/stark-rxcj2248-v4_g395m-f290lp_2478_3.spec.fits",
    z=6.1052,
)
result2 = jwspecfit.fit_lines(spec2, z=6.1052)

print(f"χ²/dof = {result2.chi2:.2f}")
for name, lr in result2.lines.items():
    if lr.snr > 1:
        print(f"  {name:<18s} flux={lr.flux:.2e} ± {lr.flux_err:.2e}  SNR={lr.snr:.1f}")

fig = jwspecfit.plot_fit(result2)
plt.show()

## Interactive plot

Use the plotly interactive plot to zoom into the Hα+[NII] complex
and inspect narrow vs broad components.